<a href="https://colab.research.google.com/github/wwilsonworld/spotify-churn/blob/main/Spotify_Churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook is aimed to practice binaray convolutional neural network using the spotify churn problem using the spotify churn dataset from kaggle.

Dataset name "Spotify Analysis Dataset 2025"

Source to dataset https://www.kaggle.com/datasets/nabihazahid/spotify-dataset-for-churn-analysis

Dataset summary: Each row represents a unique Spotify user. The dataset is aimed at evaluating churn rate where each column contains factors that would make a user churn.



Downlaod the data from a datasource. In this case we are using Kaggle.


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nabihazahid/spotify-dataset-for-churn-analysis")

print("Path to dataset files:", path)

100%|██████████| 96.8k/96.8k [00:00<00:00, 49.7MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/nabihazahid/spotify-dataset-for-churn-analysis/versions/2


Import necessary packages

In [ ]:
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt



Farmiliarize yourself with the data by viewing it (seeing the tables and graphs or the data)

In [ ]:
import os

os.listdir(path)




['spotify_churn_dataset.csv']

In [ ]:
import pandas

csv_path = os.path.join(path, "spotify_churn_dataset.csv")
data = pd.read_csv(csv_path)

In [ ]:
df = pd.DataFrame(data)
df.head()

,user_id,gender,age,country,subscription_type,listening_time,songs_played_per_day,skip_rate,device_type,ads_listened_per_week,offline_listening,is_churned
0,1,Female,54,CA,Free,26,23,0.20,Desktop,31,0,1
1,2,Other,33,DE,Family,141,62,0.34,Web,0,1,0
2,3,Male,38,AU,Premium,199,38,0.04,Mobile,0,1,1
3,4,Female,22,CA,Student,36,2,0.31,Mobile,0,1,0
4,5,Other,29,US,Family,250,57,0.36,Mobile,0,1,1


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_id                8000 non-null   int64  
 1   gender                 8000 non-null   object 
 2   age                    8000 non-null   int64  
 3   country                8000 non-null   object 
 4   subscription_type      8000 non-null   object 
 5   listening_time         8000 non-null   int64  
 6   songs_played_per_day   8000 non-null   int64  
 7   skip_rate              8000 non-null   float64
 8   device_type            8000 non-null   object 
 9   ads_listened_per_week  8000 non-null   int64  
 10  offline_listening      8000 non-null   int64  
 11  is_churned             8000 non-null   int64  
dtypes: float64(1), int64(7), object(4)
memory usage: 750.1+ KB


In [ ]:
df.shape

(8000, 12)

data analysis

In [ ]:

data = df.groupby(["country","skip_rate"]).agg(

    churn_count=("is_churned", "sum"),
    churn_rate=("is_churned", "mean"),
    total_users=("is_churned", "count")
)

data.head()



churn_count  churn_rate  total_users
country skip_rate                                      
AU      0.00                 4    0.363636           11
        0.01                 6    0.315789           19
        0.02                 2    0.105263           19
        0.03                 1    0.058824           17
        0.04                 7    0.388889           18

In [ ]:
data = data.reset_index()
data.head()

,country,skip_rate,churn_count,churn_rate,total_users
0,AU,0.00,4,0.363636,11
1,AU,0.01,6,0.315789,19
2,AU,0.02,2,0.105263,19
3,AU,0.03,1,0.058824,17
4,AU,0.04,7,0.388889,18


In [ ]:
data = df.groupby(["subscription_type"]).agg(
    churn_rate = ('is_churned','mean'),
    churn_count = ('is_churned','sum'),
    total_users = ('is_churned','count'),
).reset_index()

data.shape

(4, 4)

In [ ]:
data.head()

,subscription_type,churn_rate,churn_count,total_users
0,Family,0.275157,525,1908
1,Free,0.249257,503,2018
2,Premium,0.250591,530,2115
3,Student,0.261868,513,1959


In [ ]:
data = df.groupby(["subscription_type","songs_played_per_day"]).agg(
    churn_rate = ('is_churned','mean'),
    churn_count = ('is_churned','sum'),
    total_users = ('is_churned','count'),
).reset_index()

data.shape

(396, 5)

In [ ]:
def churned(columns):

    results = {}

    for column in columns:
        data = df.groupby(column).agg(
            churn_rate=('is_churned', 'mean')
        )

        results[column] = data

    return results

In [ ]:
churned(["subscription_type"])

{'subscription_type':                    churn_rate
 subscription_type            
 Family               0.275157
 Free                 0.249257
 Premium              0.250591
 Student              0.261868}

In [ ]:
from sklearn.preprocessing import OneHotEncoder

#create encoder
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output = False
)
#get categorical columns
categorical_cols = df.select_dtypes(include=["object"]).columns

#transform the categorical data
encoded_array = encoder.fit_transform(df[categorical_cols])

#convert into dataframe
encoded_df = pd.DataFrame(
    encoded_array,
    columns = encoder.get_feature_names_out(categorical_cols)

)

#combine numerical and encoded df together
df_final = pd.concat([df.drop(columns=categorical_cols),encoded_df],
                     axis=1)


In [ ]:
df_final.head()

,user_id,age,listening_time,songs_played_per_day,skip_rate,ads_listened_per_week,offline_listening,is_churned,gender_Female,gender_Male,...,country_PK,country_UK,country_US,subscription_type_Family,subscription_type_Free,subscription_type_Premium,subscription_type_Student,device_type_Desktop,device_type_Mobile,device_type_Web
0,1,54,26,23,0.20,31,0,1,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,2,33,141,62,0.34,0,1,0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3,38,199,38,0.04,0,1,1,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,4,22,36,2,0.31,0,1,0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4,5,29,250,57,0.36,0,1,1,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [ ]:
df_final.sample()

,user_id,age,listening_time,songs_played_per_day,skip_rate,ads_listened_per_week,offline_listening,is_churned,gender_Female,gender_Male,...,country_PK,country_UK,country_US,subscription_type_Family,subscription_type_Free,subscription_type_Premium,subscription_type_Student,device_type_Desktop,device_type_Mobile,device_type_Web
4877,4878,26,237,33,0.16,0,1,0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


Split the data into test and train split


In [ ]:
X = df_final.drop(columns=["is_churned"])
y = df_final["is_churned"]

In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
y_train

,is_churned
4661,0
5195,0
7123,0
3764,0
6824,0
...,...
5178,0
384,1
1468,0
4335,1


Scale the data

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_test)
X_test_scaled = scaler.fit_transform(X_test)




In [ ]:
X_train_scaled.shape


(1600, 25)

In [ ]:
X_test_scaled.shape

(1600, 25)

In [ ]:
y_train.shape


(6400,)

In [ ]:
y_test.shape

(1600,)

prepare the model

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(10),
    tf.keras.layers.Dense(20,activation="relu"),
    tf.keras.layers.Dense(20,activation="relu"),
    tf.keras.layers.Dense(20,activation="relu"),
    tf.keras.layers.Dense(20,activation="relu"),
    tf.keras.layers.Dense(20,activation="relu"),
    tf.keras.layers.Dense(20,activation="relu"),
    tf.keras.layers.Dense(1,activation="sigmoid")
])

model.compile(
    loss = "binary_crossentropy",
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics = ["accuracy"]
)

In [ ]:
model.fit(X_train,y_train,epochs=50)

Epoch 1/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.6389 - loss: 3.6280
Epoch 2/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7123 - loss: 0.6412
Epoch 3/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6948 - loss: 0.6780
Epoch 4/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7168 - loss: 0.6227
Epoch 5/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7324 - loss: 0.6075
Epoch 6/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7377 - loss: 0.5860
Epoch 7/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7374 - loss: 0.5916
Epoch 8/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7399 - loss: 0.5871
Epoch 9/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7466 - loss: 0.5842
Epoch 10/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7406 - loss: 0.5943
Epoch 11/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7406 - loss: 0.5839
Epoch 12/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step